# Ch.7 — Probability & Statistics

**Running theme.** Every free-throw outcome is a *draw*, not a number. Bernoulli tells us if it went in. Gaussian tells us by how much we missed. MLE turns data into parameter estimates. And the CLT explains why everything ends up looking Gaussian anyway.

**Learning outcomes.**
1. Simulate Bernoulli, Binomial, and Gaussian draws and match sample statistics to theory.
2. Watch the Central Limit Theorem kick in live — change batch size, see the sample-mean histogram morph into a Gaussian.
3. Maximise a Gaussian likelihood numerically and confirm $\hat\mu, \hat\sigma^2$ match closed-form MLE.
4. Minimise sum-of-squared-errors on a free-throw dataset and confirm the result equals the Gaussian MLE — the headline derivation from §7.

## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, FloatText, IntSlider, Dropdown, HBox, VBox, Output, jslink

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (7.5, 5.0), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})
np.set_printoptions(precision=4, suppress=True)

def linked_pair(label, value, vmin, vmax, step=0.05):
    sl = FloatSlider(value=value, min=vmin, max=vmax, step=step,
                     description=label, continuous_update=True, readout=False)
    tx = FloatText(value=value, description=" ", step=step,
                   layout={"width": "110px"})
    jslink((sl, "value"), (tx, "value"))
    return sl, tx

rng = np.random.default_rng(2026)

## 2 · Sample from the three workhorses and check moments

For each distribution, draw $10^5$ samples and compare empirical mean/variance to theory.

In [ ]:
N = 100_000

# Bernoulli(p=0.72): does the shot go in?
p = 0.72
s_bern = rng.binomial(1, p, size=N)
print(f"Bernoulli(p={p}):  mean {s_bern.mean():.4f} (theory {p})   "
      f"var {s_bern.var():.4f} (theory {p*(1-p):.4f})")

# Binomial(n=10, p=0.72): makes in 10 shots
n_trials = 10
s_bin = rng.binomial(n_trials, p, size=N)
print(f"Binomial(n={n_trials}, p={p}): mean {s_bin.mean():.4f} (theory {n_trials*p})   "
      f"var {s_bin.var():.4f} (theory {n_trials*p*(1-p):.4f})")

# Gaussian(mu=2.10, sigma=0.05): release height in metres
mu, sigma = 2.10, 0.05
s_norm = rng.normal(mu, sigma, size=N)
print(f"Gaussian(μ={mu}, σ={sigma}): mean {s_norm.mean():.5f} (theory {mu})   "
      f"var {s_norm.var():.6f} (theory {sigma**2:.6f})")

## 3 · Interactive CLT — morph any skewed source into a Gaussian by averaging

Pick the source distribution and the batch size $n$. The widget draws 10 000 batch means and shows the histogram vs the CLT prediction $\mathcal{N}(\mu,\, \sigma^2/n)$.

Start with $n=1$ (raw distribution, possibly skewed) and watch it become Gaussian as $n$ rises.

In [ ]:
src_dd = Dropdown(options=["exponential", "uniform", "bernoulli", "bimodal"],
                  value="exponential", description="source")
n_sl  = IntSlider(value=1, min=1, max=200, step=1, description="batch n",
                  continuous_update=True)
reps_sl = IntSlider(value=10_000, min=500, max=20_000, step=500,
                    description="reps", continuous_update=False)
readout = Output()

fig, ax = plt.subplots(figsize=(7.0, 4.5))

def draw_source(kind, size):
    if kind == "exponential":  return rng.exponential(1.0, size=size), 1.0, 1.0
    if kind == "uniform":      return rng.uniform(0, 1, size=size), 0.5, 1/12
    if kind == "bernoulli":    return (rng.random(size=size) < 0.3).astype(float), 0.3, 0.3*0.7
    if kind == "bimodal":
        mask = rng.random(size=size) < 0.5
        samps = np.where(mask, rng.normal(-1, 0.3, size=size),
                               rng.normal(+1, 0.3, size=size))
        return samps, 0.0, 1.0 + 0.09    # approximate moments

def update_clt(*_):
    reps, n = reps_sl.value, n_sl.value
    samples, mu_src, var_src = draw_source(src_dd.value, (reps, n))
    means = samples.mean(axis=1)
    ax.clear()
    ax.hist(means, bins=60, density=True, color="#27AE60", alpha=0.65,
            edgecolor="#2C3E50", linewidth=0.4)
    mu_clt, sd_clt = mu_src, np.sqrt(var_src / n)
    xs = np.linspace(means.min(), means.max(), 300)
    pred = (1 / (sd_clt * np.sqrt(2*np.pi))) * np.exp(-0.5*((xs - mu_clt)/sd_clt)**2)
    ax.plot(xs, pred, color="#E74C3C", lw=2.4,
            label=fr"CLT: $\mathcal{{N}}({mu_clt:.3f},\,{sd_clt:.3f}^2)$")
    ax.set_title(f"{src_dd.value} source, batch size n={n}")
    ax.set_xlabel("batch mean"); ax.set_ylabel("density")
    ax.legend(loc="upper right"); ax.grid(alpha=0.25)
    with readout:
        readout.clear_output(wait=True)
        print(f"empirical mean = {means.mean():.4f}   (theory {mu_clt:.4f})")
        print(f"empirical std  = {means.std():.4f}   (CLT {sd_clt:.4f})")
    fig.canvas.draw_idle()

for w in (src_dd, n_sl, reps_sl):
    w.observe(update_clt, names="value")
update_clt()

display(VBox([HBox([src_dd, n_sl, reps_sl]), readout]))

## 4 · MLE for a Gaussian — closed form vs grid search

Generate 200 samples from $\mathcal{N}(\mu,\sigma^2)$ with known parameters; then (a) compute the closed-form MLE $\hat\mu = \bar y$, $\hat\sigma^2 = \tfrac1N \sum (y_i - \bar y)^2$, and (b) scan a 2-D grid of $(\mu, \sigma)$ values maximising $\log \mathcal{L}$. The two must agree.

In [ ]:
mu_true, sigma_true = 2.10, 0.12
y = rng.normal(mu_true, sigma_true, size=200)

# Closed form
mu_hat = y.mean()
sigma_hat = np.sqrt(((y - mu_hat) ** 2).mean())
print(f"closed-form MLE:  μ̂ = {mu_hat:.4f}   σ̂ = {sigma_hat:.4f}")

# Grid search on log-likelihood
mu_grid = np.linspace(1.95, 2.25, 301)
sig_grid = np.linspace(0.06, 0.20, 301)
M, S = np.meshgrid(mu_grid, sig_grid)
loglik = -0.5 * np.log(2*np.pi*S**2) * len(y) \
         - ((y[:, None, None] - M[None, :, :]) ** 2).sum(axis=0) / (2 * S**2)
idx = np.unravel_index(np.argmax(loglik), loglik.shape)
print(f"grid-search MLE:  μ̂ = {M[idx]:.4f}   σ̂ = {S[idx]:.4f}")

# Plot the log-likelihood surface
fig3, ax3 = plt.subplots(figsize=(6.5, 4.2))
cf = ax3.contourf(M, S, loglik, levels=25, cmap="Purples")
ax3.plot(mu_hat, sigma_hat, "*", color="#F39C12", markersize=18,
         markeredgecolor="#2C3E50", label="closed-form MLE")
ax3.plot(mu_true, sigma_true, "o", color="#E74C3C", markersize=10,
         markeredgecolor="#2C3E50", label="truth")
ax3.set_xlabel("μ"); ax3.set_ylabel("σ"); ax3.legend()
ax3.set_title("Gaussian log-likelihood surface")
plt.show()

## 5 · Headline result — MLE under Gaussian noise == least squares

Free-throw arc with Gaussian measurement noise. We'll fit the parabola two ways and confirm they return the same weights:

1. **MSE minimisation** — minimise $\sum_i (y_i - X_i \mathbf{w})^2$ with `np.linalg.lstsq`.
2. **MLE** — maximise $\log \mathcal{L}(\mathbf{w}, \sigma)$ by closed form.

They must coincide. If they don't, §7 of the README is a lie — which it isn't.

In [ ]:
# Simulate noisy free-throw samples.
v0y, h0, g = 7.2, 2.10, 9.81
t = np.linspace(0.05, 1.40, 25)
y_clean = h0 + v0y * t - 0.5 * g * t ** 2
y_obs = y_clean + rng.normal(0, 0.05, size=t.shape)
X = np.column_stack([np.ones_like(t), t, t ** 2])

# Method 1: OLS
w_mse, *_ = np.linalg.lstsq(X, y_obs, rcond=None)

# Method 2: MLE — closed form gives the SAME w_mse, plus σ̂².
w_mle = np.linalg.solve(X.T @ X, X.T @ y_obs)
resid = y_obs - X @ w_mle
sigma2_mle = (resid ** 2).mean()

print(f"OLS weights : {w_mse}")
print(f"MLE weights : {w_mle}")
print(f"max |OLS - MLE|  = {np.max(np.abs(w_mse - w_mle)):.2e}  (should be ≈ 0)")
print(f"MLE noise σ̂    = {np.sqrt(sigma2_mle):.4f}   (true σ = 0.05)")

## 6 · Bernoulli MLE — free-throw success rate with confidence interval

Observed: 72 makes out of 100 shots. What is the MLE for $p$ and a 95% confidence interval?

In [ ]:
makes, total = 72, 100
p_hat = makes / total
se = np.sqrt(p_hat * (1 - p_hat) / total)   # standard error from CLT
lo, hi = p_hat - 1.96 * se, p_hat + 1.96 * se

print(f"MLE p̂           = {p_hat}")
print(f"standard error   = {se:.4f}")
print(f"95% CI (normal)  = [{lo:.4f}, {hi:.4f}]")
print()
print("Sanity: if the 'true' p is 0.65, how often do 100-shot CIs cover it?")
reps = 5000
p_true = 0.65
outcomes = rng.binomial(total, p_true, size=reps) / total
ses = np.sqrt(outcomes * (1 - outcomes) / total)
covered = np.sum((outcomes - 1.96*ses <= p_true) & (p_true <= outcomes + 1.96*ses))
print(f"coverage over {reps} repeats: {covered/reps*100:.2f}%   (target 95%)")

## 7 · Pick a noise model, get a loss

Quick demo that switching from Gaussian to Laplace noise switches the loss from L2 to L1.

In [ ]:
# For a 1-D estimation task with y_i = μ + ε_i:
#   ε ~ Gaussian  ⇒  MLE of μ  is the mean
#   ε ~ Laplace   ⇒  MLE of μ  is the median

# Generate data with a few hard outliers.
clean = rng.normal(0.0, 1.0, size=200)
outliers = np.array([15.0, -12.0, 20.0, -18.0])
y = np.concatenate([clean, outliers])

mu_gauss_mle = y.mean()         # minimiser of sum of squares  == Gaussian MLE
mu_laplace_mle = np.median(y)   # minimiser of sum of |·|      == Laplace MLE

print(f"sample size: {len(y)}  (200 clean + 4 outliers)")
print(f"Gaussian MLE (mean)   = {mu_gauss_mle:+.4f}  ← pulled by outliers")
print(f"Laplace  MLE (median) = {mu_laplace_mle:+.4f}  ← robust")
print("\nThat's why robust regression swaps MSE → MAE when outliers are suspected.")

## 8 · Where this reappears

- **ML Ch.1 Linear Regression.** Cell 5 is Ch.1's derivation in miniature.
- **ML Ch.2 Logistic Regression.** Cell 6's Bernoulli MLE generalised to $p_i = \sigma(\mathbf{x}_i^\top \mathbf{w})$.
- **ML Ch.15 MLE & Loss Functions.** The full catalogue extending cell 7.
- **ML Ch.4 Neural Networks.** Softmax + cross-entropy = multinomial MLE.
- **AI Ch.6 Bayesian inference.** Promote point estimates $\hat\theta$ to posterior distributions $p(\theta \mid \text{data})$.

**Final exercise.** Repeat cell 6 but compute the **Bayesian posterior** for $p$ using a Beta(2, 2) prior. Plot the posterior density and compare its mode to the MLE. Which estimator do you trust more with only $n=10$ shots? With $n=1000$?